# Water Quality Prediction — Final Random Forest Notebook (Snowflake Strict)

This notebook is formatted to **strictly match Snowflake Notebook JSON conventions** (every cell has an `id` and `metadata.name`).

It improves the benchmark Random Forest approach (~0.20) by focusing on generalisation:

1. **Station-aware cross validation** using `GroupKFold` (groups derived from lat/lon; lat/lon are **NOT** features).
2. **Feature expansion**: use all numeric Landsat + TerraClimate predictors available + stable engineered indices.
3. **RandomForest tuning + regularisation** to reduce overfit.

✅ Model family: **RandomForestRegressor only**.

## 0) Install dependencies (Snowflake)
Run once, then restart kernel (Connected → Restart kernel).

In [ ]:
!pip install uv
!uv pip install -r '../requirements.txt'

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

from sklearn.feature_selection import SelectKBest, mutual_info_regression

from IPython.display import display

## 1) Load data
Expected files (same as benchmark package):
- `water_quality_training_dataset.csv`
- `landsat_features_training.csv`
- `terraclimate_features_training.csv`
- `submission_template.csv`
- `landsat_features_validation.csv`
- `terraclimate_features_validation.csv`

In [ ]:
Water_Quality_df = pd.read_csv('../water_quality_training_dataset.csv')
landsat_train = pd.read_csv('../landsat_features_training_full.csv')
terraclimate_train = pd.read_csv('../terraclimate_features_training.csv')

print('Water_Quality_df:', Water_Quality_df.shape)
print('landsat_train:', landsat_train.shape)
print('terraclimate_train:', terraclimate_train.shape)

display(Water_Quality_df.head())
display(landsat_train.head())
display(terraclimate_train.head())

## 2) Join + define targets
We join columns, drop duplicates, and keep lat/lon only for grouping (NOT features).

In [ ]:
TARGET_COLS = ['Total Alkalinity','Electrical Conductance','Dissolved Reactive Phosphorus']
LAT_CANDS = ['Latitude','latitude','LAT','lat']
LON_CANDS = ['Longitude','longitude','LON','lon']
DATE_CANDS = ['Sample Date','sample_date','date','Date']

wq = pd.concat([Water_Quality_df, landsat_train, terraclimate_train], axis=1)
wq = wq.loc[:, ~wq.columns.duplicated()]

for col in ['NDMI','MNDWI']:
    if col in wq.columns:
        wq[col] = pd.to_numeric(wq[col], errors='coerce')

lat_col = next((c for c in LAT_CANDS if c in wq.columns), None)
lon_col = next((c for c in LON_CANDS if c in wq.columns), None)

print('lat_col:', lat_col, '| lon_col:', lon_col)

y_TA = wq[TARGET_COLS[0]].copy()
y_EC = wq[TARGET_COLS[1]].copy()
y_DRP = wq[TARGET_COLS[2]].copy()

## 3) Feature engineering (rules-compliant)
- Convert Sample Date into seasonality features (month/dayofyear + sin/cos)
- Drop lat/lon from predictors
- Keep numeric predictors only
- Add stable indices if source bands exist

In [ ]:
EPS = 1e-9

X_all = wq.drop(columns=[c for c in TARGET_COLS if c in wq.columns], errors='ignore').copy()

# Date features
sample_date_col = next((c for c in DATE_CANDS if c in X_all.columns), None)
if sample_date_col is not None:
    dt = pd.to_datetime(X_all[sample_date_col], errors='coerce', dayfirst=True)
    X_all['month'] = dt.dt.month.astype('float')
    X_all['dayofyear'] = dt.dt.dayofyear.astype('float')
    X_all['sin_doy'] = np.sin(2*np.pi*(X_all['dayofyear']/365.25))
    X_all['cos_doy'] = np.cos(2*np.pi*(X_all['dayofyear']/365.25))
    X_all.drop(columns=[sample_date_col], inplace=True)

# Drop lat/lon from features
if lat_col in X_all.columns:
    X_all.drop(columns=[lat_col], inplace=True)
if lon_col in X_all.columns:
    X_all.drop(columns=[lon_col], inplace=True)

X = X_all.select_dtypes(include=[np.number]).copy()

band_map = {
    'nir': ['nir','NIR'],
    'green': ['green','Green'],
    'red': ['red','Red'],
    'swir16': ['swir16','SWIR16','swir_16'],
    'swir22': ['swir22','SWIR22','swir_22'],
}

def find_col(df, cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

nir = find_col(X, band_map['nir'])
green = find_col(X, band_map['green'])
red = find_col(X, band_map['red'])
swir16 = find_col(X, band_map['swir16'])
swir22 = find_col(X, band_map['swir22'])

if nir and green:
    X['NDWI_green_nir'] = (X[green] - X[nir]) / (X[green] + X[nir] + EPS)
if nir and red:
    X['NDVI_nir_red'] = (X[nir] - X[red]) / (X[nir] + X[red] + EPS)
if nir and swir22:
    X['NBR_nir_swir22'] = (X[nir] - X[swir22]) / (X[nir] + X[swir22] + EPS)
if swir22 and swir16:
    X['swir22_swir16_ratio'] = X[swir22] / (X[swir16] + EPS)

print('Feature matrix shape:', X.shape)
display(X.head())

## 4) Station-aware grouping for CV
We group rows by station (approx) using lat/lon rounded to 3 decimals.
Lat/lon are used only to create **groups**, not as features.

In [ ]:
if lat_col is None or lon_col is None:
    groups = np.arange(len(wq))
    print('WARNING: lat/lon not found; grouping falls back to row-wise groups.')
else:
    lat = pd.to_numeric(wq[lat_col], errors='coerce')
    lon = pd.to_numeric(wq[lon_col], errors='coerce')
    groups = (lat.round(3).astype(str) + '_' + lon.round(3).astype(str)).fillna('nan_nan').values

print('Unique station-groups:', pd.Series(groups).nunique())

## 5) Optional target transform (log1p)
Switch off if it hurts CV.

In [ ]:
USE_LOG1P = True

def tform(y):
    y = pd.to_numeric(y, errors='coerce')
    if USE_LOG1P:
        return np.log1p(np.clip(y, a_min=0, a_max=None))
    return y.values

def inv_tform(yhat):
    if USE_LOG1P:
        return np.expm1(yhat)
    return yhat

yTA = tform(y_TA)
yEC = tform(y_EC)
yDRP = tform(y_DRP)

print('USE_LOG1P:', USE_LOG1P)

## 6) Pipeline + tuning (GroupKFold)
Includes optional feature selection to reduce noise.

In [ ]:
USE_FEATURE_SELECTION = True
TOP_K = 60
k = min(TOP_K, X.shape[1])

steps = [('imputer', SimpleImputer(strategy='median'))]
if USE_FEATURE_SELECTION:
    steps.append(('select', SelectKBest(score_func=mutual_info_regression, k=k)))
steps.append(('rf', RandomForestRegressor(random_state=42, n_jobs=-1)))

pipe = Pipeline(steps)

param_dist = {
    'rf__n_estimators': [800, 1200, 1600, 2000],
    'rf__max_depth': [None, 20, 40, 60],
    'rf__min_samples_split': [2, 5, 10, 20],
    'rf__min_samples_leaf': [1, 2, 4, 8],
    'rf__max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
    'rf__bootstrap': [True],
    'rf__max_samples': [0.6, 0.8, 1.0],
}

cv = GroupKFold(n_splits=5)

def tune_one_target(Xmat, yvec, groups, label):
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_dist,
        n_iter=15,
        scoring='r2',
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=2
    )
    search.fit(Xmat, yvec, groups=groups)
    print(f'[{label}] best CV R2: {search.best_score_:.4f}')
    print(f'[{label}] best params: {search.best_params_}')
    return search.best_estimator_

print('USE_FEATURE_SELECTION:', USE_FEATURE_SELECTION, '| k:', k)

## 7) Tune models (TA / EC / DRP)

In [ ]:
best_TA = tune_one_target(X, yTA, groups, 'TA')
best_EC = tune_one_target(X, yEC, groups, 'EC')
best_DRP = tune_one_target(X, yDRP, groups, 'DRP')

## 8) Group holdout sanity check

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, yTA, groups=groups))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]

yTA_tr, yTA_te = yTA[train_idx], yTA[test_idx]
yEC_tr, yEC_te = yEC[train_idx], yEC[test_idx]
yDRP_tr, yDRP_te = yDRP[train_idx], yDRP[test_idx]

best_TA.fit(X_tr, yTA_tr)
best_EC.fit(X_tr, yEC_tr)
best_DRP.fit(X_tr, yDRP_tr)

pTA = inv_tform(best_TA.predict(X_te))
pEC = inv_tform(best_EC.predict(X_te))
pDRP = inv_tform(best_DRP.predict(X_te))

true_TA = y_TA.iloc[test_idx].values
true_EC = y_EC.iloc[test_idx].values
true_DRP = y_DRP.iloc[test_idx].values

r2_TA = r2_score(true_TA, pTA)
r2_EC = r2_score(true_EC, pEC)
r2_DRP = r2_score(true_DRP, pDRP)

print('Holdout (group-aware) mean R2:', round(float(np.mean([r2_TA, r2_EC, r2_DRP])), 4))

## 9) Refit on all training rows

In [ ]:
best_TA.fit(X, yTA)
best_EC.fit(X, yEC)
best_DRP.fit(X, yDRP)
print('Refit complete.')

## 10) Prepare validation features (same engineering + align columns)

In [ ]:
submission_template = pd.read_csv('../submission_template.csv')
landsat_val = pd.read_csv('../landsat_features_validation.csv')
terraclimate_val = pd.read_csv('../terraclimate_features_validation.csv')

val = pd.concat([landsat_val, terraclimate_val], axis=1)
val = val.loc[:, ~val.columns.duplicated()]

for col in ['NDMI','MNDWI']:
    if col in val.columns:
        val[col] = pd.to_numeric(val[col], errors='coerce')

val_date_col = next((c for c in DATE_CANDS if c in val.columns), None)
if val_date_col is not None:
    dt = pd.to_datetime(val[val_date_col], errors='coerce', dayfirst=True)
    val['month'] = dt.dt.month.astype('float')
    val['dayofyear'] = dt.dt.dayofyear.astype('float')
    val['sin_doy'] = np.sin(2*np.pi*(val['dayofyear']/365.25))
    val['cos_doy'] = np.cos(2*np.pi*(val['dayofyear']/365.25))
    val.drop(columns=[val_date_col], inplace=True)

for c in LAT_CANDS + LON_CANDS:
    if c in val.columns:
        val.drop(columns=[c], inplace=True)

Xv = val.select_dtypes(include=[np.number]).copy()

nir_v = find_col(Xv, band_map['nir'])
green_v = find_col(Xv, band_map['green'])
red_v = find_col(Xv, band_map['red'])
swir16_v = find_col(Xv, band_map['swir16'])
swir22_v = find_col(Xv, band_map['swir22'])

if nir_v and green_v:
    Xv['NDWI_green_nir'] = (Xv[green_v] - Xv[nir_v]) / (Xv[green_v] + Xv[nir_v] + EPS)
if nir_v and red_v:
    Xv['NDVI_nir_red'] = (Xv[nir_v] - Xv[red_v]) / (Xv[nir_v] + Xv[red_v] + EPS)
if nir_v and swir22_v:
    Xv['NBR_nir_swir22'] = (Xv[nir_v] - Xv[swir22_v]) / (Xv[nir_v] + Xv[swir22_v] + EPS)
if swir22_v and swir16_v:
    Xv['swir22_swir16_ratio'] = Xv[swir22_v] / (Xv[swir16_v] + EPS)

for c in X.columns:
    if c not in Xv.columns:
        Xv[c] = np.nan
Xv = Xv[X.columns]

print('Validation X shape:', Xv.shape)

## 11) Predict + export submission

In [ ]:
pred_TA = inv_tform(best_TA.predict(Xv))
pred_EC = inv_tform(best_EC.predict(Xv))
pred_DRP = inv_tform(best_DRP.predict(Xv))

submission_df = pd.DataFrame({
    'Longitude': submission_template['Longitude'].values,
    'Latitude': submission_template['Latitude'].values,
    'Sample Date': submission_template['Sample Date'].values,
    'Total Alkalinity': pred_TA,
    'Electrical Conductance': pred_EC,
    'Dissolved Reactive Phosphorus': pred_DRP,
})

display(submission_df.head())

submission_df.to_csv('/tmp/submission_rf_blended.csv', index=False)
print('Saved /tmp/submission_rf_blended.csv')

In [ ]:
put_sql = """
PUT file:///tmp/submission_rf_blended.csv
'snow://workspace/USER$.PUBLIC."snowflake-challenge"/versions/live/submission'
AUTO_COMPRESS=FALSE
OVERWRITE=TRUE
"""

session.sql(put_sql).collect()
print('File saved! Refresh the browser to see the files in the sidebar')